# OpenCaseLaw API - Example Notebook

This notebook demonstrates the `opencaselaw` Python package for the public no-key OpenCaseLaw REST API.

The package is independently developed and is not official, endorsed by, associated with, or affiliated with OpenCaseLaw.

Most cells call the live public API. Keep the default rate limit enabled and avoid rerunning the whole notebook repeatedly in tight loops.

## 1. Setup and Installation

In [1]:
# Run `uv sync` from the repo root and select the repo `.venv` as this notebook's kernel.
import opencaselaw

print(f"opencaselaw {opencaselaw.__version__} ready")

opencaselaw 0.1.0 ready


In [2]:
from pprint import pprint

from opencaselaw import OpenCaseLawClient

client = OpenCaseLawClient()


def preview(value, keys=None, limit=3):
    """Print compact previews of dictionaries, lists, and model objects."""
    if hasattr(value, "raw"):
        value = value.raw
    if isinstance(value, dict):
        selected = value if keys is None else {key: value.get(key) for key in keys}
        pprint(selected)
    elif isinstance(value, list):
        print(f"{len(value)} items")
        for item in value[:limit]:
            preview(item, keys=keys, limit=limit)
    else:
        print(value)


DEFAULT_DECISION_ID = "bger_4A_747_2012"
DEFAULT_BGE_REF = "BGE 140 III 86"

## 2. Basic API Information

In [3]:
print(f"Base URL: {client.base_url}")
print(f"Timeout: {client.timeout}s")
print(f"Rate limit delay: {client.rate_limit_delay}s")

Base URL: https://mcp.opencaselaw.ch/api
Timeout: 30.0s
Rate limit delay: 0.2s


### 2.1 Courts

In [4]:
courts = client.list_courts()
preview(courts)

122 items
{'canton': 'CH',
 'court': 'bger',
 'decision_count': 175684,
 'earliest': '1986-10-06',
 'languages': 3,
 'latest': '2026-05-11'}
{'canton': 'GE',
 'court': 'ge_gerichte',
 'decision_count': 168026,
 'earliest': '1908-11-08',
 'languages': 3,
 'latest': '2026-04-28'}
{'canton': 'CH',
 'court': 'bvger',
 'decision_count': 92448,
 'earliest': '1951-07-28',
 'languages': 3,
 'latest': '2026-05-13'}


### 2.2 Statistics and Scraper Health

In [5]:
statistics = client.get_statistics(court="bger")
preview(statistics)

{'by_court': {'bger': 175684},
 'by_language': {'de': 107564, 'fr': 58062, 'it': 10058},
 'by_year': {'2007': 7511,
             '2008': 7293,
             '2009': 7055,
             '2010': 7234,
             '2011': 7128,
             '2012': 7396,
             '2013': 7527,
             '2014': 7233,
             '2015': 7396,
             '2016': 7455,
             '2017': 7452,
             '2018': 7783,
             '2019': 7665,
             '2020': 7510,
             '2021': 7254,
             '2022': 6886,
             '2023': 7098,
             '2024': 7036,
             '2025': 7493,
             '2026': 2577},
 'total': 175684}


In [6]:
health = client.get_scraper_health()
preview(health)

{'disk': {'data_volume': {'free_gb': 125.4,
                          'path': '/mnt/HC_Volume_104655575',
                          'total_gb': 350.0,
                          'used_gb': 224.5,
                          'used_percent': 64.2},
          'repo': {'free_gb': 67.7,
                   'path': '/opt/caselaw/repo',
                   'total_gb': 149.9,
                   'used_gb': 76.1,
                   'used_percent': 50.8}},
 'run_at': '2026-05-23T02:44:14.126615+00:00',
 'run_duration_s': 6253.8,
 'run_source': 'cron',
 'scrapers': {'ag_gerichte': {'duration_s': 594.7,
                              'error': None,
                              'error_count': 0,
                              'gap': -1,
                              'new_count': 9,
                              'none_count': 0,
                              'note': None,
                              'our_count': 10489,
                              'portal_count': 10488,
                              'sk

## 3. Case Law Search

### 3.1 Simple Search

In [7]:
results = client.search_decisions(query="Mietrecht Kündigung", limit=5)

print(f"Found {results.total} results; showing {len(results.results)}")
for decision in results.results:
    print(decision.decision_id, decision.citation_string_de, decision.decision_date)
    print(" ", decision.title)
    print(" ", decision.canonical_url)

Found 108 results; showing 5
bge_BGE_125_III_231 BGE 125 III 231 1999-01-01
  None
  https://mcp.opencaselaw.ch/entscheid/bge_BGE_125_III_231
bge_BGE_117_II_410 BGE 117 II 410 1991-01-01
  None
  https://mcp.opencaselaw.ch/entscheid/bge_BGE_117_II_410
bger_4A_747_2012 BGer 4A_747/2012 vom 5. April 2013 2013-04-05
  Mietrecht; Kündigung,
  https://mcp.opencaselaw.ch/entscheid/bger_4A_747_2012
bge_BGE_119_II_147 BGE 119 II 147 1993-01-01
  None
  https://mcp.opencaselaw.ch/entscheid/bge_BGE_119_II_147
bge_BGE_122_III_262 BGE 122 III 262 1996-01-01
  None
  https://mcp.opencaselaw.ch/entscheid/bge_BGE_122_III_262


### 3.2 Search with Filters and Sorting

In [8]:
filtered = client.search_decisions(
    query="Arbeitsvertrag Kündigung",
    court="bger",
    canton="CH",
    language="de",
    date_from="2020-01-01",
    date_to="2024-12-31",
    limit=5,
    sort="date_desc",
    fields="compact",
)

for decision in filtered.results:
    print(decision.decision_date, decision.citation_string_de, decision.title)

2024-12-10 BGer 9C 378/2024 vom 10. Dezember 2024 None
2024-12-03 BGer 4A_463/2024 vom 3. Dezember 2024 None
2024-11-08 BGer 4A_353/2024 vom 8. November 2024 None
2024-10-11 BGer 4A 268/2024 vom 11. Oktober 2024 None
2024-09-05 BGer 4D_103/2024 vom 5. September 2024 None


### 3.3 Pagination

In [9]:
page_size = 3
page_1 = client.search_decisions(query="Datenschutz", limit=page_size, offset=0)
page_2 = client.search_decisions(query="Datenschutz", limit=page_size, offset=page_size)

print("Page 1")
for decision in page_1.results:
    print(" ", decision.decision_id)

print("Page 2")
for decision in page_2.results:
    print(" ", decision.decision_id)

Page 1
  sz_verwaltungsgericht_III 2020 75
  edoeb_2017_-_Datenschutz_bei_Windows10__Schlussbericht_
  edoeb_Betreiber_m_ssen_bei_der_Erfassung_der_Kontaktdaten_Datenschutz_sicherstellen
Page 2
  edoeb_Presse_80318
  edoeb_Presse_84210
  edoeb_Presse_60675


### 3.4 Fetch a Single Decision

In [10]:
decision_id = results.results[0].decision_id if results.results else DEFAULT_DECISION_ID
decision = client.get_decision(decision_id, full_text=True)

print(decision.decision_id)
print(decision.court, decision.decision_date, decision.language)
print(decision.title)
print((decision.full_text or "")[:500])

bge_BGE_125_III_231
bge 1999-01-01 de
None
Bundesgericht (BGE) Band III 1999 BGE 125 III 231
Tribunal fédéral (ATF) Volume III 1999 BGE 125 III 231
Tribunale federale (DTF) Volume III 1999 BGE 125 III 231

Regeste
 Mietrecht; Kündigungsschutz für einen zusammen mit einer Wohnung vermieteten Autoabstellplatz; Untersuchungsmaxime bei mietrechtlichen Streitigkeiten (Art. 253a, 266l, 269d, 274d Abs. 3 OR). Begriff der mitvermieteten Sache im Sinne von Art. 253a Abs. 1 OR (E. 2). Bei der Kündigung von formell separat mitvermieteten Sachen dur


## 4. Decision Structure

In [11]:
structure = client.get_structure(decision_id, paragraph_excerpt_chars=300)
preview(structure)

{'_note': 'Erwägungen-paragraphs are returned as excerpts; call '
          'get_erwaegung(decision_id, e_number) for the verbatim full text of '
          'a specific paragraph.',
 'canonical_url': 'https://mcp.opencaselaw.ch/entscheid/bge_BGE_125_III_231',
 'court': 'bge',
 'decision_date': '1999-04-13',
 'decision_id': 'bge_BGE_125_III_231',
 'dispositiv': None,
 'dispositiv_orders': [],
 'erwaegungen_paragraph_count': 3,
 'erwaegungen_paragraphs': [{'depth': 1,
                             'e_number': '2',
                             'parent': None,
                             'text_chars': 3424,
                             'text_excerpt': 'Gesondert vermietete '
                                             'Einstellplätze können unter '
                                             'Einhaltung einer zweiwöchigen '
                                             'Frist jeweils auf Ende eines '
                                             'Monats gekündigt werden (\n'
               

In [12]:
regeste = client.get_regeste(decision_id)
preview(regeste)

{'_note': 'The Regeste is the official court-formulated summary of the legal '
          "rule. References like '(E. 5.2.1)' inside the Regeste point to "
          'specific Erwägungen — use get_erwaegung(decision_id, e_number) to '
          'retrieve their verbatim text. The returned `regeste` has every '
          'inner Swiss-case reference pre-wrapped as a Markdown link — quote '
          'it verbatim to the user.',
 'canonical_url': 'https://mcp.opencaselaw.ch/entscheid/bge_BGE_125_III_231',
 'citation_string_de': 'BGE 125 III 231',
 'citation_string_fr': 'ATF 125 III 231',
 'citation_string_it': 'DTF 125 III 231',
 'court': 'bge',
 'decision_date': '1999-04-13',
 'decision_id': 'bge_BGE_125_III_231',
 'language': 'de',
 'markdown_link': '[BGE 125 III '
                  '231](https://mcp.opencaselaw.ch/entscheid/bge_BGE_125_III_231)',
 'regeste': '<br/>\n'
            'Regeste\n'
            '<br/>Mietrecht; Kündigungsschutz für einen zusammen mit einer '
            'Wohnung 

In [13]:
try:
    erwaegung = client.get_erwaegung(decision_id, "2.3")
    preview(erwaegung)
except Exception as error:
    print(f"This decision may not have Erwägung 2.3: {error}")

{'available_e_numbers': ['2', '3', '4'],
 'error': "E. '2.3' not found in 'bge_BGE_125_III_231'."}


In [14]:
relevant = client.find_relevant_erwaegung(
    decision_id,
    claim="Die Kündigung eines Mietvertrags muss nach Treu und Glauben erfolgen.",
    max_paragraphs=3,
)
preview(relevant)

{'_hint': 'No Erwägung clearly matched the claim (BM25 gap < 1.2). '
          'best_low_confidence_match holds the rank-1 result — do NOT cite it '
          'as the relevant Erwägung. Tell the user no Erwägung clearly matches '
          'and ask for a more specific claim, or list the top-k as candidates '
          'without picking.',
 'best_low_confidence_match': {'citation_string_de': 'BGE 125 III 231, E. 3',
                               'citation_string_fr': 'ATF 125 III 231, consid. '
                                                     '3',
                               'citation_string_it': 'DTF 125 III 231, consid. '
                                                     '3',
                               'depth': 1,
                               'display_url': 'https://mcp.opencaselaw.ch/entscheid/bge_BGE_125_III_231#e-3?highlight=270b%20OR%0A%29%20wie%20auch%20die%20Pr%C3%BCfung%2C%20ob%20die%20damit%20ausgesprochene%20K%C3%BCndigung%20des%20bisherigen%20Mietvertrages%20

## 5. Citation Graph and Citation Integrity

In [15]:
citation = client.cite(DEFAULT_BGE_REF, pinpoint="2.3", language="de")
print(citation.exists)
print(citation.citation_string_de)
print(citation.canonical_url)
print(citation.rule_statement)

True
BGE 140 III 86, E. 2.3
https://mcp.opencaselaw.ch/entscheid/bge_BGE_140_III_86#e-2-3
Regeste
 Art. 42 Abs. 2 BGG, Art. 18 Abs. 1 und Art. 32 Abs. 1 OR; Pflicht zur Begründung der Rechtsverletzungen; Willenserklärung durch einen Vertreter. Anforderungen an die Begründung, welche die Verfahrensparteien zu erfüllen haben (E. 2). Auslegung des Willens des Vertreters, der dem Vertretenen zugerechnet wird (E. 4).
Regeste
 Art. 42 al. 2 LTF, art. 18 al. 1 et art. 32 al. […]


In [16]:
citations = client.get_citations(
    decision_id, direction="both", min_confidence=0.3, limit=5
)
preview(citations)

{'decision_id': 'bge_BGE_125_III_231',
 'direction': 'both',
 'incoming': [{'confidence_score': 0.75,
               'court': 'ge_gerichte',
               'decision_date': '2026-04-07',
               'docket_number': 'ATAS/304/2026',
               'mention_count': 1,
               'source_decision_id': 'ge_gerichte_ATAS_304_2026',
               'target_ref': '125 III 231'},
              {'confidence_score': 0.75,
               'court': 'vd_gerichte',
               'decision_date': '2026-03-23',
               'docket_number': 'XZ24.018379',
               'mention_count': 1,
               'source_decision_id': 'vd_gerichte_XZ24.018379',
               'target_ref': '125 III 231'},
              {'confidence_score': 0.75,
               'court': 'ge_gerichte',
               'decision_date': '2026-03-19',
               'docket_number': 'ATAS/273/2026',
               'mention_count': 1,
               'source_decision_id': 'ge_gerichte_ATAS_273_2026',
               'target_re

In [17]:
appeal_chain = client.get_appeal_chain(decision_id, min_confidence=0.3)
preview(appeal_chain)

{'chain': [],
 'court': 'bge',
 'decision_date': '1999-01-01',
 'decision_id': 'bge_BGE_125_III_231',
 'docket_number': 'BGE 125 III 231'}


In [18]:
leading_cases = client.find_leading_cases(
    query="Mietrecht Kündigung", court="bger", limit=5
)
preview(leading_cases)

{'article': None,
 'law_code': None,
 'query': 'Mietrecht Kündigung',
 'results': [{'canonical_url': 'https://mcp.opencaselaw.ch/entscheid/bger_4A_565_2017',
              'citation_count': 165,
              'citation_string_de': 'BGer 4A 565/2017 vom 11. Juli 2018',
              'citation_string_fr': 'TF 4A 565/2017 du 11 juillet 2018',
              'citation_string_it': 'TF 4A 565/2017 del 11 luglio 2018',
              'court': 'bger',
              'decision_date': '2018-07-11',
              'decision_id': 'bger_4A_565_2017',
              'docket_number': '4A 565/2017',
              'markdown_link': '[4A '
                               '565/2017](https://mcp.opencaselaw.ch/entscheid/bger_4A_565_2017)',
              'pinpoint': {'confidence': 'medium',
                           'e_number': '1.2.2.3',
                           'matched_sentence': 'Gemäss dem Grundgedanken für '
                                               'die Streitwertberechnung, '
                     

In [19]:
trends = client.analyze_trends(query="Datenschutz", court="bger")
preview(trends)

{'article': None,
 'law_code': None,
 'query': 'Datenschutz',
 'total': 457,
 'years': [{'count': 4, 'year': 2000},
           {'count': 14, 'year': 2001},
           {'count': 7, 'year': 2002},
           {'count': 6, 'year': 2003},
           {'count': 26, 'year': 2004},
           {'count': 6, 'year': 2005},
           {'count': 11, 'year': 2006},
           {'count': 12, 'year': 2007},
           {'count': 5, 'year': 2008},
           {'count': 8, 'year': 2009},
           {'count': 11, 'year': 2010},
           {'count': 17, 'year': 2011},
           {'count': 12, 'year': 2012},
           {'count': 8, 'year': 2013},
           {'count': 14, 'year': 2014},
           {'count': 21, 'year': 2015},
           {'count': 23, 'year': 2016},
           {'count': 17, 'year': 2017},
           {'count': 19, 'year': 2018},
           {'count': 32, 'year': 2019},
           {'count': 36, 'year': 2020},
           {'count': 34, 'year': 2021},
           {'count': 26, 'year': 2022},
          

In [20]:
attestation = client.attest(
    draft_text="Gemäss BGE 140 III 86 E. 2.3 ist die Kündigung zu prüfen."
)
preview(attestation)

{'_note': 'Closing audit covers up to FIVE hallucination classes:\n'
          '  • case      — citation exists in corpus, pinpoint resolves\n'
          '  • statute   — Art. X LAW reference resolves in statutes.db\n'
          '  • quote     — "…"-text appears verbatim in a cited source\n'
          "  • date      — 'vom DD.MM.YYYY' adjacent to citation matches\n"
          '  • grounding — (opt-in via audit_grounding=True) the proposition\n'
          '               attached to each verified citation is actually\n'
          '               supported by the cited Erwägung / Regeste / text.\n'
          "               Closes the 'reasoning error' class identified by\n"
          "               Butler & Butler, 'Legal RAG Bench' (Isaacus, 2026):\n"
          '               citation correct + source retrieved + proposition\n'
          '               unsupported. Costs one Sonnet call (~3 s, ≈$0.005)\n'
          '               regardless of citation count.\n'
          '\n'
    

In [21]:
claim_check = client.verify_claim(
    claim="Die Kündigung eines Mietvertrags darf nicht rechtsmissbräuchlich sein.",
    decision_id=decision_id,
)
preview(claim_check)

{'_note': 'Verified by an independent Sonnet judge against verbatim text. If '
          'supports=no|contradicts|unrelated, DO NOT use this decision to '
          'support this claim — either find a different authority or qualify '
          'your statement.',
 'canonical_url': 'https://mcp.opencaselaw.ch/entscheid/bge_BGE_125_III_231',
 'checked_text_source': 'Regeste',
 'citation_string_de': 'BGE 125 III 231',
 'citation_string_fr': 'ATF 125 III 231',
 'citation_string_it': 'DTF 125 III 231',
 'claim': 'Die Kündigung eines Mietvertrags darf nicht rechtsmissbräuchlich '
          'sein.',
 'confidence': 0.85,
 'decision_id': 'bge_BGE_125_III_231',
 'qualifying_excerpt': None,
 'reasoning': 'The text addresses form requirements and co-leased objects, not '
              'the prohibition of abusive termination (Rechtsmissbrauch).',
 'supporting_excerpt': None,
 'supports': 'unrelated'}


## 6. Statutes and Legislation

In [22]:
law_hits = client.search_laws(
    "Schadenersatz", canton="CH", jurisdiction="federal", limit=5
)
preview(law_hits)

{'cantonal_hits': 0,
 'count': 5,
 'federal_hits': 5,
 'query': '(schadenersatz OR dommages OR indemnite OR risarcimento OR '
          'schadensersatz)',
 'results': [{'abbreviation': '?',
              'article_num': '§ 13',
              'canton': 'CH',
              'heading': 'Responsabilité et dommages-intérêts',
              'level': 'federal',
              'snippet': '...3 Celui qui est limité de manière injustifiée et '
                         'grave dans sa liberté personnelle, a droit à des '
                         '>>>dommages<<<-intérêts et à une >>>indemnité<<< '
                         'pour tort moral.\n'
                         '4 En cas d’expropriation ou de restriction '
                         'importante à la propriété, une...',
              'sr_number': '131.222.2',
              'title': 'Verfassung des Kantons Basel-Landschaft, vom 17. Mai '
                       '1984'},
             {'abbreviation': '?',
              'article_num': '30',
           

In [23]:
law = client.get_law("OR", article="41", language="de")
print(law.abbreviation, law.sr_number, law.title)
for article in law.articles:
    print(article.article_num, article.heading)
    print(article.text[:800])

OR 220 Bundesgesetz vom 30. März 1911 betreffend die Ergänzung des Schweizerischen Zivilgesetzbuches (Fünfter Teil: Obligationenrecht)
41 None
1 Wer einem andern widerrechtlich Schaden zufügt, sei es mit Absicht, sei es aus Fahrlässigkeit, wird ihm zum Ersatze verpflichtet.
2 Ebenso ist zum Ersatze verpflichtet, wer einem andern in einer gegen die guten Sitten verstossenden Weise absichtlich Schaden zufügt.


In [24]:
amendment = client.resolve_amendment_ref(ref_type="BBl", year=2020, page=1)
preview(amendment)

{'eli_uri': None}


In [25]:
legislation_hits = client.search_legislation(
    query="Datenschutz",
    canton="CH",
    limit=5,
    active_only=True,
    search_in_content=False,
)
preview(legislation_hits)

{'canton': 'CH',
 'language': 'all (DE+FR+IT)',
 'laws': [{'category': 'Verordnung',
           'entity': 'CH',
           'entity_name': 'Bund',
           'is_active': True,
           'keywords': 'Datenschutzverordnung, DSV',
           'lexfind_id': 26101,
           'original_url': 'https://www.fedlex.admin.ch/eli/cc/2022/568/de',
           'snippet': '',
           'systematic_number': '235.11',
           'title': 'Verordnung über den Datenschutz',
           'version_active_since': '01.12.2025'},
          {'category': 'Gesetz',
           'entity': 'CH',
           'entity_name': 'Bund',
           'is_active': True,
           'keywords': 'Datenschutzgesetz, DSG',
           'lexfind_id': 25084,
           'original_url': 'https://www.fedlex.admin.ch/eli/cc/2022/491/de',
           'snippet': '',
           'systematic_number': '235.1',
           'title': 'Bundesgesetz über den Datenschutz',
           'version_active_since': '01.04.2025'},
          {'category': 'Anderes',

In [26]:
# Use a LexFind ID from the previous result if the response exposes one.
lexfind_id = None
items = legislation_hits.get("results") or legislation_hits.get("items") or []
if items and isinstance(items[0], dict):
    lexfind_id = items[0].get("lexfind_id") or items[0].get("id")

if lexfind_id is not None:
    legislation = client.get_legislation(int(lexfind_id), include_versions=False)
    preview(legislation)
else:
    print("No LexFind ID found in the search response preview.")

No LexFind ID found in the search response preview.


In [27]:
changes = client.get_legislation_changes(canton="CH")
preview(changes)

{'canton': 'CH',
 'changes': [{'category': 'Staatsvertrag',
              'change_date': '23.05.2026',
              'change_type': 'formless',
              'entity': 'CH',
              'entity_name': 'Bund',
              'is_active': True,
              'lexfind_id': 26956,
              'original_url': 'https://www.fedlex.admin.ch/eli/cc/1998/2734_2734_2734/de',
              'systematic_number': '0.730.0',
              'title': 'Vertrag über die Energiecharta'},
             {'category': 'Verordnung',
              'change_date': '23.05.2026',
              'change_type': 'version',
              'entity': 'CH',
              'entity_name': 'Bund',
              'is_active': True,
              'lexfind_id': 34300,
              'original_url': 'https://www.fedlex.admin.ch/eli/cc/2023/335/de',
              'systematic_number': '946.231.156.5',
              'title': 'Verordnung über Massnahmen betreffend Moldau'},
             {'category': 'Verordnung',
              'change_da

## 7. Commentaries, Doctrine, and Materials

In [28]:
commentary_hits = client.search_commentaries(
    "Schadenersatz", abbreviation="OR", language="de", limit=5
)
preview(commentary_hits)

{'count': 5,
 'query': 'Schadenersatz',
 'results': [{'abbreviation': 'OR',
              'article_num': '97',
              'authors': ['Clarisse von Wunschheim', 'Cristina Wullschleger'],
              'html_link': 'https://onlinekommentar.ch/de/kommentare/or97',
              'language': 'de',
              'snippet': '...Der fällige >>>Schadenersatz<<< darf den '
                         'tatsächlichen Wert des entstandenen Schadens nicht '
                         'übersteigen. Insbesondere kennt das Schweizer Recht '
                         'den Begriff des „Strafschadensersatzes“, wie er in '
                         'bestimmten Common-Law-Rechtsordnungen bekannt ist, '
                         'nicht. Das Schweizer Recht erlaubt es den Parteien '
                         'jedoch, vertragliche...',
              'sr_number': '220',
              'title': 'Art. 97 OR'},
             {'abbreviation': 'OR',
              'article_num': '97',
              'authors': ['Clarisse von

In [29]:
commentary = client.get_commentary("OR", article="41", language="de")
preview(commentary)

{'article': '41', 'error': 'No commentary found for Art. 41.', 'law': 'OR'}


In [30]:
doctrine = client.get_doctrine("Art. 41 OR Schadenersatz")
preview(doctrine)

{'commentary': None,
 'doctrine_summary': {'authority': '6323 citations',
                      'coverage_decades': {'1990s': 2,
                                           '2000s': 2,
                                           '2010s': 3,
                                           '2020s': 1},
                      'established_by': 'BGE 126 I 97',
                      'note': 'Doctrine has evolved across 7 distinct '
                              'holdings. Review the timeline for shifts in '
                              'court reasoning.',
                      'principal_rule': 'Art',
                      'total_citations': 13940,
                      'total_leading_cases': 8},
 'doctrine_timeline': [{'bge_ref': 'BGE 123 III 110',
                        'rule_added': 'Haftung des Motorfahrzeughalters',
                        'year': '1997'},
                       {'bge_ref': 'BGE 125 IV 161',
                        'rule_added': 'Legitimation des Geschädigten zur '
         

In [31]:
materials = client.search_materialien("Haftung", law_code="OR", limit=5)
preview(materials)

{'count': 0,
 'debate_results': [{'council': 'nationalrat',
                     'law_code': 'BV',
                     'page': 301,
                     'snippet': '...Unser Begehren ist, die >>>Haftung<<< für '
                                'Betreiber von Atomanla-\n'
                                'gen in der Verfassung zu verankern. Wir '
                                'bitten Sie um Unter-\n'
                                'stützung. Vielleicht hilft Ihnen ein Wort, '
                                'das uns Herr Direktor\n'
                                'Koller vom Bundesamt für Justiz in der '
                                'Kommission mitgege-\n'
                                'ben...',
                     'source': 'Amtliches Bulletin'},
                    {'council': 'nationalrat',
                     'law_code': 'BV',
                     'page': 302,
                     'snippet': '...In den meisten Bereichen des geltenden '
                                'Rec

In [32]:
article_materials = client.get_materialien("OR", article="41")
preview(article_materials)

{'error': 'No Materialien found for OR Art. 41. Try a different law or check '
          'get_statistics.'}


In [33]:
purpose = client.get_article_purpose("220", "41", language="de", max_paragraphs=5)
preview(purpose)

{'_hint': 'No direct article→Botschaft link available; matches are FTS5 '
          'co-occurrences of the SR number and article reference inside the '
          'verbatim corpus. Quote with care — verify the snippet is actually '
          'discussing the article in question, not just naming it in passing.',
 'article': '41',
 'language': 'de',
 'sources': [{'bbl_citation': 'BBl 2025 3622',
              'eli_uri': 'https://fedlex.data.admin.ch/eli/fga/2025/3622',
              'format': 'akoma-ntoso-xml',
              'paragraphs': [{'page': None,
                              'section': None,
                              'text': 'Im Zusammenhang mit dem Rahmenkredit '
                                      '2021–2027 wurde das Risikomanagement '
                                      'der EGW in der '
                                      'Wohnraumförderungsverordnung vom 26. '
                                      'November 2003 (WFV) verankert (Art. 41 '
                          

In [34]:
botschaft_hits = client.search_botschaft("Schadenersatz", language="de", limit=5)
preview(botschaft_hits)

{'language_filter': 'de',
 'query': 'Schadenersatz',
 'results': [{'article_anchor': None,
              'bbl_citation': 'BBl 2023 1460',
              'eli_uri': 'https://fedlex.data.admin.ch/eli/fga/2023/1460',
              'language': 'de',
              'page': None,
              'publication_date': None,
              'section': None,
              'snippet': 'Das ersuchte Gericht kann die Anerkennung oder '
                         'Vollstreckung einer Entscheidung versagen, sofern '
                         'und soweit mit ihr <<<Schadenersatz>>> zugesprochen '
                         'wird, der eine Partei nicht für…'},
             {'article_anchor': '28',
              'bbl_citation': 'BBl 2025 3702',
              'eli_uri': 'https://fedlex.data.admin.ch/eli/fga/2025/3702',
              'language': 'de',
              'page': None,
              'publication_date': None,
              'section': None,
              'snippet': 'Erlässt das Schiedsgericht einen endgültigen

In [35]:
history = client.get_article_history("220", "41", language="de", leading_cases_limit=5)
preview(history)

{'_hint': 'Timeline is ordered chronologically. Each entry has a `kind` field '
          '(botschaft | court_decision | commentary) and a stable URI. Use '
          'get_article_purpose for verbatim Botschaft text, find_citations for '
          'the full citation network of a court decision in the timeline.',
 'article': '41',
 'language': 'de',
 'sr_number': '220',
 'statute': {'article': '41',
             'consolidation_date': '2026-01-01',
             'current_text': '1 Wer einem andern widerrechtlich Schaden '
                             'zufügt, sei es mit Absicht, sei es aus '
                             'Fahrlässigkeit, wird ihm zum Ersatze '
                             'verpflichtet.\n'
                             '2 Ebenso ist zum Ersatze verpflichtet, wer einem '
                             'andern in einer gegen die guten Sitten '
                             'verstossenden Weise absichtlich Schaden zufügt.',
             'language': 'de',
             'law_abbrevi

## 8. Research Helpers

In [36]:
case_brief = client.get_case_brief(DEFAULT_BGE_REF)
preview(case_brief)

{'_extraction_quality': 'structured (high)',
 '_hint': 'For verbatim text of a specific Erwägung, call '
          'get_erwaegung(decision_id, e_number).',
 'authority': {'incoming_citations': 11096, 'outgoing_citations': 42},
 'bge_ref': 'BGE 140 III 86',
 'canonical_url': 'https://mcp.opencaselaw.ch/entscheid/bge_BGE_140_III_86',
 'citation_string_de': 'BGE 140 III 86',
 'citation_string_fr': 'ATF 140 III 86',
 'citation_string_it': 'DTF 140 III 86',
 'court': 'bge',
 'date': '2014-01-23',
 'decision_id': 'bge_BGE_140_III_86',
 'dispositiv': '',
 'key_erwaegungen': [{'depth': 1,
                      'e_number': '2',
                      'text': "Le Tribunal fédéral applique le droit d'office "
                              '(\n'
                              'art. 106 al. 1 LTF\n'
                              "). Toutefois, compte tenu de l'obligation de "
                              'motiver qui incombe au recourant en vertu de '
                              "l'\n"
           

In [37]:
exam_question = client.generate_exam_question("Vertragsrecht")
preview(exam_question)

{'analysis': {'applicable_statutes': ['BGG 105', 'BGG 106', 'CP 146'],
              'correct_outcome': '2)',
              'leading_case': 'BGE 147 IV 73',
              'legal_test': 'Regeste\n Art'},
 'difficulty': 3,
 'fact_pattern': 'Bundesgericht (BGE) Band IV 08.01.2021 BGE 147 IV 73 '
                 '(6B_572/2020)\n'
                 'Tribunal fédéral (ATF) Volume IV 08.01.2021 BGE 147 IV 73 '
                 '(6B_572/2020)\n'
                 'Tribunale federale (DTF) Volume IV 08.01.2021 BGE 147 IV 73 '
                 '(6B_572/2020)\n'
                 '\n'
                 'Regeste\n'
                 ' Art. 146 Abs. 1 StGB; Betrug; Entgelt für sexuelle '
                 'Dienstleistungen; Täuschung über die Zahlungsbereitschaft; '
                 'Arglist; Vermögensschaden. Die Vorspiegelung der '
                 'Zahlungsbereitschaft ist als Täuschung über innere Tatsachen '
                 'grundsätzlich arglistig. Dass das Täuschungsopfer im '
                 '

In [38]:
# Research-only: sends facts/question to the public API; do not use confidential facts or treat output as legal advice.
mock = client.mock_decision(
    facts="Eine Mieterin kündigt nach einer strittigen Nebenkostenabrechnung fristlos.",
    question="Welche zivilrechtlichen Fragen stellen sich?",
    preferred_language="de",
    limit=3,
    request_timeout=120.0,
)
preview(mock)

{'applicable_statutes': [],
 'clarification_answers': [],
 'clarification_gate': {'required_high_priority': ['timeline_dates',
                                                   'procedural_posture',
                                                   'requested_relief'],
                        'status': 'needs_clarification',
                        'unanswered_high_priority': ['timeline_dates',
                                                     'procedural_posture',
                                                     'requested_relief']},
 'clarifying_questions': [{'id': 'timeline_dates',
                           'priority': 'high',
                           'question': 'What are the key dates (administrative '
                                       'decision, service date, appeal filing '
                                       'date)?',
                           'why_it_matters': 'Admissibility and deadline '
                                             'checks depend on exac

## 9. Exports and Feeds

In [39]:
docx_bytes = client.export_docx(decision_id)
pdf_bytes = client.export_pdf(decision_id)
bib_text = client.export_bib(decision_id)
ris_text = client.export_ris(decision_id)

print(f"DOCX bytes: {len(docx_bytes):,}")
print(f"PDF bytes: {len(pdf_bytes):,}")
print(bib_text[:500])
print(ris_text[:500])

DOCX bytes: 45,178
PDF bytes: 19,242
@misc{bgeBGE125III231,
  title         = {BGE 125 III 231},
  author        = {Bundesgericht (BGE)},
  year          = {1999},
  howpublished  = {OpenCaseLaw — opencaselaw.ch},
  url           = {https://mcp.opencaselaw.ch/entscheid/bge_BGE_125_III_231},
  note          = {Docket: BGE 125 III 231}
}

TY  - CASE
TI  - BGE 125 III 231
AU  - Bundesgericht (BGE)
PY  - 1999
DA  - 1999/01/01/
AN  - BGE 125 III 231
LA  - de
AB  - Regeste
 Mietrecht; Kündigungsschutz für einen zusammen mit einer Wohnung vermieteten Autoabstellplatz; Untersuchungsmaxime bei mietrechtlichen Streitigkeiten (Art. 253a, 266l, 269d, 274d Abs. 3 OR). Begriff der mitvermieteten Sache im Sinne von Art. 253a Abs. 1 OR (E. 2). Bei der Kündigung von formell separat mitvermieteten Sachen durch den Vermieter genügt es, das


In [40]:
# Uncomment to save exports locally. Generated files are intentionally not tracked.
from pathlib import Path

output_dir = Path("downloads")
output_dir.mkdir(exist_ok=True)
(output_dir / f"{decision_id}.docx").write_bytes(docx_bytes)
(output_dir / f"{decision_id}.pdf").write_bytes(pdf_bytes)
(output_dir / f"{decision_id}.bib").write_text(bib_text, encoding="utf-8")
(output_dir / f"{decision_id}.ris").write_text(ris_text, encoding="utf-8")

2233

In [41]:
feed = client.atom_feed("bger")
print(feed[:1000])

<?xml version="1.0" encoding="utf-8"?>
<feed xmlns="http://www.w3.org/2005/Atom">
  <title>Bundesgericht — neue Entscheide</title>
  <link rel="self" type="application/atom+xml" href="https://mcp.opencaselaw.ch/atom/bger.xml"/>
  <link rel="alternate" type="text/html" href="https://mcp.opencaselaw.ch/courts/bger"/>
  <updated>2026-05-23T11:22:09Z</updated>
  <id>https://mcp.opencaselaw.ch/atom/bger.xml</id>
  <author><name>OpenCaseLaw</name></author>
  <subtitle>Daily-refreshed feed of newly published decisions. CC0 — opencaselaw.ch</subtitle>
  <entry>
    <id>https://mcp.opencaselaw.ch/entscheid/bger_7B_465_2026</id>
    <title>BGer 7B_465/2026 vom 11. Mai 2026</title>
    <link rel="alternate" type="text/html" href="https://mcp.opencaselaw.ch/entscheid/bger_7B_465_2026"/>
    <updated>2026-05-11T00:00:00Z</updated>
    <published>2026-05-11T00:00:00Z</published>
  </entry>
  <entry>
    <id>https://mcp.opencaselaw.ch/entscheid/bger_7B_480_2026</id>
    <title>BGer 7B_480/2026 vom 11

## 10. Integrity Proof

In [42]:
integrity = client.get_integrity_proof(decision_id)
preview(integrity)

{'algorithm': 'RFC6962-SHA256',
 'date': '2026-05-22',
 'decision_id': 'bge_BGE_125_III_231',
 'leaf_encoding': 'decision_id\\ncli_ch\\necli\\ncontent_hash\\ndecision_date',
 'leaf_hash': '18f0d81b2c601793902b75c68c4e87c4c9026fe968fdac1440f26fc01efe53b3',
 'leaf_index': 61223,
 'ots_proof_url': '/integrity/2026-05-22.root.ots',
 'proof': [{'position': 'L',
            'sibling_hash': 'fffc2c647277a44314f2888b3435b2796bf5bc6ac6aa1a524002426cad5711a0'},
           {'position': 'L',
            'sibling_hash': 'eefcb216beeb9e98f115a694eccf478a1dc0214bb557068772a2258690a9f4a3'},
           {'position': 'L',
            'sibling_hash': '6a92e52980bc511bcf88b6ead05ed45e75f9590d2ea5ca0109fabddad83afda7'},
           {'position': 'R',
            'sibling_hash': '0d7bbc2a62b1cbb323758d8dda3635f76fd1ac2f253b69cc4daa24f48a592f8f'},
           {'position': 'R',
            'sibling_hash': 'b43ff1a17d818b22c31f636672f3de4800b0aec9e879093c478bb31e1ac42967'},
           {'position': 'L',
           

## 11. Context Manager and Rate Limiting

In [43]:
with OpenCaseLawClient(timeout=60.0, rate_limit_delay=0.2) as managed_client:
    managed_results = managed_client.search_decisions(
        query="Bundesgericht",
        court="bger",
        limit=3,
        fields="compact",
    )
    print(f"Found {managed_results.total} results")

print("Client closed after leaving the context manager")

Found 63 results
Client closed after leaving the context manager


In [44]:
# Long-running batch jobs should keep the default 0.2s delay or use a larger value.
slow_client = OpenCaseLawClient(timeout=30.0, rate_limit_delay=1.0)
try:
    sample = slow_client.search_decisions(query="Test", limit=1)
    print(f"Sample result count: {sample.total}")
finally:
    slow_client.close()

Sample result count: 60


## Summary

This notebook demonstrates:

1. Case law search, filters, pagination, and single-decision lookup
2. Decision structure, Regeste, and Erwägung helpers
3. Citation graph, canonical citation, attestation, and claim verification
4. Statutes, legislation, commentaries, doctrine, and materials
5. Research helpers, exports, Atom feeds, statistics, health, and integrity proofs

Use rate limiting for batch jobs and avoid sending confidential text to public API endpoints.

In [45]:
client.close()